In [1]:
import pandas as pd
import numpy as np
from dowhy import CausalModel

import pytimetk as tk
from missingno import matrix

import statsmodels.api as am
from statsmodels.genmod.generalized_linear_model import GLM
from statsmodels.genmod.families import Binomial
from statsmodels.genmod.families.links import logit
from sklearn.metrics import roc_auc_score

In [2]:
df_raw = pd.read_csv('../data/raw/bank.csv')

In [3]:
display(df_raw)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,33,blue-collar,single,primary,no,1,yes,no,cellular,20,apr,257,1,-1,0,unknown,no
11158,39,services,married,secondary,no,733,no,no,unknown,16,jun,83,4,-1,0,unknown,no
11159,32,technician,single,secondary,no,29,no,no,cellular,19,aug,156,2,-1,0,unknown,no
11160,43,technician,married,secondary,no,0,no,yes,cellular,8,may,9,2,172,5,failure,no


In [4]:
df_new = pd.read_csv('../data/processed/bank_selected.csv')

In [5]:
display(df_new)

,log_balance,balance_shifted,age,age_squared,was_contacted_before,has_debt,financial_stress,treatment,previous,net_balance_indicator,...,month_mar,month_oct,month_sep,month_dec,marital_single,education_tertiary,marital_married,pdays,deposit,deposit_numeric
0,9.125980,9191,59,3481,0,1,0,0,0,1,...,False,False,False,False,False,False,True,-1,yes,1
1,8.838262,6893,56,3136,0,0,0,0,0,1,...,False,False,False,False,False,False,True,-1,yes,1
2,9.001839,8118,41,1681,0,1,0,0,0,1,...,False,False,False,False,False,False,True,-1,yes,1
3,9.140347,9324,55,3025,0,1,0,0,0,1,...,False,False,False,False,False,False,True,-1,yes,1
4,8.858226,7032,54,2916,0,0,0,0,0,1,...,False,False,False,False,False,True,True,-1,yes,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,8.831858,6849,33,1089,0,1,0,1,0,1,...,False,False,False,False,True,False,False,-1,no,0
11158,8.933400,7581,39,1521,0,0,0,0,0,1,...,False,False,False,False,False,False,True,-1,no,0
11159,8.835938,6877,32,1024,0,0,0,1,0,1,...,False,False,False,False,True,False,False,-1,no,0
11160,8.831712,6848,43,1849,1,1,1,1,5,0,...,False,False,False,False,False,False,True,172,no,0


# Krok 3 — Analiza wpływu transformacji

Dla każdej transformacji wykonanej w dataset.md:
1. Uruchom model z tą transformacją i bez niej
2. Porównaj AUC / F1 / feature importance
3. Odpowiedz: czy transformacja poprawia model, usuwa sygnał, narusza relację przyczynową
4. Zaloguj wynik do raportu

---

Poniżej będą wykonywane testy transformacji: binarne *_bin, logarytmy, bucketowanie, pivoty, sezonowość, status materialny.

In [8]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

# Wczytaj dane
df_raw = pd.read_csv('../data/raw/bank.csv')
df_new = pd.read_csv('../data/processed/bank_data_cleaned.csv')

# Target binarny
if 'deposit' in df_new.columns:
    y = df_new['deposit']
elif 'deposit_numeric' in df_new.columns:
    y = df_new['deposit_numeric']
else:
    raise ValueError("Brak kolumny 'deposit' w df_new!")

# Lista transformacji do testu (dostosuj do kolumn df_new)
# Automatyczna detekcja transformacji na podstawie nazw kolumn
sufiksy = ['_log1p', '_log', '_bin', '_capped', '_squared', '_indicator']
transformacje = [col for col in df_new.columns if any(s in col for s in sufiksy) and col != 'deposit']

print("Wykryte transformacje:", transformacje)

# Baza bez transformacji + bez targetu
X_base_full = df_new.drop(columns=transformacje + ['deposit', 'deposit_numeric'], errors='ignore')

wyniki_transformacji = []

for t in transformacje:
    # BASE (bez transformacji)
    X_base = X_base_full.copy()

    # Z transformacją (jedną!)
    X_tr = X_base_full.copy()
    X_tr[t] = df_new[t]

    # Encoding
    X_base_enc = pd.get_dummies(X_base, drop_first=True)
    X_tr_enc = pd.get_dummies(X_tr, drop_first=True)

    # Align kolumn
    X_base_enc, X_tr_enc = X_base_enc.align(X_tr_enc, join='outer', axis=1, fill_value=0)
    X_base_enc.columns = X_base_enc.columns.astype(str)
    X_tr_enc.columns = X_tr_enc.columns.astype(str)
    X_base_enc.columns = X_base_enc.columns.str.replace(r'[\\[\\]<]', '', regex=True)
    X_tr_enc.columns = X_tr_enc.columns.str.replace(r'[\\[\\]<]', '', regex=True)

    # TEN SAM SPLIT!
    X_train_b, X_test_b, y_train, y_test = train_test_split(
        X_base_enc, y, test_size=0.3, random_state=42, stratify=y
    )
    X_train_t, X_test_t, _, _ = train_test_split(
        X_tr_enc, y, test_size=0.3, random_state=42, stratify=y
    )

    # MODELE
    model_b = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
    model_t = xgb.XGBClassifier(eval_metric='logloss', random_state=42)
    model_b.fit(X_train_b, y_train)
    model_t.fit(X_train_t, y_train)

    # PREDYKCJE
    y_pred_b = model_b.predict(X_test_b)
    y_pred_t = model_t.predict(X_test_t)
    y_proba_b = model_b.predict_proba(X_test_b)[:, 1]
    y_proba_t = model_t.predict_proba(X_test_t)[:, 1]

    # METRYKI
    try:
        auc_b = roc_auc_score(y_test, y_proba_b)
        auc_t = roc_auc_score(y_test, y_proba_t)
    except:
        auc_b, auc_t = np.nan, np.nan
    f1_b = f1_score(y_test, y_pred_b)
    f1_t = f1_score(y_test, y_pred_t)

    # feature importance
    feat_imp_tr = pd.Series(
        model_t.feature_importances_, index=X_tr_enc.columns
    ).sort_values(ascending=False)

    # LOG
    wyniki_transformacji.append({
        'transformacja': t,
        'auc_base': auc_b,
        'auc_tr': auc_t,
        'delta_auc': auc_t - auc_b if pd.notnull(auc_b) and pd.notnull(auc_t) else np.nan,
        'f1_base': f1_b,
        'f1_tr': f1_t,
        'delta_f1': f1_t - f1_b,
        'top_feat': feat_imp_tr.head(5).index.tolist()
    })

# wynik
wyniki_df = pd.DataFrame(wyniki_transformacji).sort_values(by='delta_auc', ascending=False)
display(wyniki_df)

Wykryte transformacje: ['previous_log1p', 'balance_log1p', 'net_balance_indicator', 'campaign_capped', 'age_squared']


,transformacja,auc_base,auc_tr,delta_auc,f1_base,f1_tr,delta_f1,top_feat
3,campaign_capped,0.745019,0.746179,0.001160,0.637423,0.645028,0.007605,"[poutcome_success, contact_is_unknown, is_peak..."
2,net_balance_indicator,0.745019,0.745341,0.000322,0.637423,0.644452,0.007029,"[poutcome_success, contact_is_unknown, is_peak..."
4,age_squared,0.745019,0.743191,-0.001828,0.637423,0.632849,-0.004574,"[poutcome_success, contact_is_unknown, is_peak..."
1,balance_log1p,0.745019,0.741066,-0.003953,0.637423,0.644254,0.006831,"[poutcome_success, contact_is_unknown, is_peak..."
0,previous_log1p,0.745019,0.740582,-0.004437,0.637423,0.626045,-0.011378,"[poutcome_success, contact_is_unknown, is_peak..."


# Krok 4 — Dobór modeli i porównanie

Używamy df_raw (oryginalny zbiór) i df_new (z cechami).
Porównujemy modele na:
- oryginalnych cechach (df_raw)
- nowych cechach (df_new, tylko zaakceptowane transformacje)

Wyniki poniżej pokazują, jak zmieniają się metryki (AUC, F1) po zastosowaniu transformacji i nowych cech względem oryginalnych danych.

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

# Przygotowanie danych oryginalnych
df_raw['deposit_numeric'] = df_raw['deposit'].map({'yes': 1, 'no': 0}) if 'deposit' in df_raw.columns else df_raw['deposit']
orig_features = [
    'age', 'job', 'contact', 'day', 'pdays', 'previous', 'poutcome',
    'has_debt', 'net_balance_indicator', 'treatment', 'financial_stress',
    'was_contacted_before', 'contact_intensity_past', 'balance_shifted',
    'balance', 'month', 'education', 'marital'
]
orig_features = [col for col in orig_features if col in df_raw.columns]
X_orig = df_raw[orig_features]
y_orig = df_raw['deposit_numeric']
X_orig_enc = pd.get_dummies(X_orig, drop_first=True)
X_train_o, X_test_o, y_train_o, y_test_o = train_test_split(X_orig_enc, y_orig, test_size=0.3, random_state=42, stratify=y_orig)

# Automatyczna detekcja transformacji w df_new
sufiksy = ['_log1p', '_log', '_bin', '_capped', '_squared', '_indicator']
transformacje = [col for col in df_new.columns if any(s in col for s in sufiksy) and col not in ['deposit', 'deposit_numeric']]

# Przygotowanie danych z nowymi cechami
features_final = orig_features + transformacje
features_final = [f for f in features_final if f in df_new.columns]
X_new = df_new[features_final]
y_new = df_new['deposit_numeric'] if 'deposit_numeric' in df_new.columns else df_new['deposit']
X_new_enc = pd.get_dummies(X_new, drop_first=True)
X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(X_new_enc, y_new, test_size=0.3, random_state=42, stratify=y_new)

# Random Forest na oryginalnych cechach
rf_o = RandomForestClassifier(random_state=42)
rf_o.fit(X_train_o, y_train_o)
y_pred_rf_o = rf_o.predict(X_test_o)
y_pred_rf_o_proba = rf_o.predict_proba(X_test_o)[:, 1]
auc_rf_o = roc_auc_score(y_test_o, y_pred_rf_o_proba)
f1_rf_o = f1_score(y_test_o, y_pred_rf_o)

# Random Forest na nowych cechach
rf_n = RandomForestClassifier(random_state=42)
rf_n.fit(X_train_n, y_train_n)
y_pred_rf_n = rf_n.predict(X_test_n)
y_pred_rf_n_proba = rf_n.predict_proba(X_test_n)[:, 1]
auc_rf_n = roc_auc_score(y_test_n, y_pred_rf_n_proba)
f1_rf_n = f1_score(y_test_n, y_pred_rf_n)

# Regresja logistyczna na oryginalnych cechach
lr_o = LogisticRegression(max_iter=1000, random_state=42)
lr_o.fit(X_train_o, y_train_o)
y_pred_lr_o = lr_o.predict(X_test_o)
y_pred_lr_o_proba = lr_o.predict_proba(X_test_o)[:, 1]
auc_lr_o = roc_auc_score(y_test_o, y_pred_lr_o_proba)
f1_lr_o = f1_score(y_test_o, y_pred_lr_o)

# Regresja logistyczna na nowych cechach
lr_n = LogisticRegression(max_iter=1000, random_state=42)
lr_n.fit(X_train_n, y_train_n)
y_pred_lr_n = lr_n.predict(X_test_n)
y_pred_lr_n_proba = lr_n.predict_proba(X_test_n)[:, 1]
auc_lr_n = roc_auc_score(y_test_n, y_pred_lr_n_proba)
f1_lr_n = f1_score(y_test_n, y_pred_lr_n)

print(f"Random Forest (oryginalne cechy): AUC={auc_rf_o:.3f}, F1={f1_rf_o:.3f}")
print(f"Random Forest (nowe cechy): AUC={auc_rf_n:.3f}, F1={f1_rf_n:.3f}")
print(f"Logistic Regression (oryginalne cechy): AUC={auc_lr_o:.3f}, F1={f1_lr_o:.3f}")
print(f"Logistic Regression (nowe cechy): AUC={auc_lr_n:.3f}, F1={f1_lr_n:.3f}")


Random Forest (oryginalne cechy): AUC=0.762, F1=0.680
Random Forest (nowe cechy): AUC=0.663, F1=0.591
Logistic Regression (oryginalne cechy): AUC=0.753, F1=0.628
Logistic Regression (nowe cechy): AUC=0.667, F1=0.567


In [11]:
print("y full:\n", y.value_counts())
print("y_train:\n", y_train.value_counts())
print("y_test:\n", y_test.value_counts())
print("classes:", rf.classes_ if 'rf' in locals() else "model not trained")

y full:
 deposit
0    5873
1    5289
Name: count, dtype: int64
y_train:
 deposit
0    4111
1    3702
Name: count, dtype: int64
y_test:
 deposit
0    1762
1    1587
Name: count, dtype: int64
classes: model not trained


In [12]:
print("Unique classes in y_train:", y_train.nunique())

Unique classes in y_train: 2


In [13]:
print("Train:", y_train.value_counts())
print("Test:", y_test.value_counts())

Train: deposit
0    4111
1    3702
Name: count, dtype: int64
Test: deposit
0    1762
1    1587
Name: count, dtype: int64


# Krok 5 — Raport końcowy

## Raport

Dla każdej transformacji:
```
Transformacja: [nazwa]
Status: ACCEPTED / REJECTED
Efekt na AUC: [przed] → [po]
Powód: [1-2 zdania]
```

**Podsumowanie odpowiedzi na pytania badawcze:**
1. **Feature engineering nie poprawił modelu vs baseline:**
   - Random Forest na oryginalnych cechach: AUC=0.762, F1=0.680
   - Random Forest na nowych cechach: AUC=0.674, F1=0.591
   - Logistic Regression na oryginalnych cechach: AUC=0.753, F1=0.628
   - Logistic Regression na nowych cechach: AUC=0.670, F1=0.553
   - Wnioski: nowe cechy i transformacje nie poprawiły skuteczności modeli, a wręcz ją obniżyły względem oryginalnych danych.
2. **Które transformacje pomogły / zaszkodziły:**
   - Szczegóły w tabeli testów transformacji (sekcja 3). Większość transformacji nie poprawiła metryk.
3. **Czy usunięto istotny sygnał:**
   - Tak — niektóre transformacje mogły usunąć sygnał, co widać po spadku AUC/F1.
4. **Top 5 najważniejszych cech końcowego modelu:**
   - Wypisane w komórce z feature importance Random Forest.
5. **Najlepiej pasujący model:**
   - Random Forest na oryginalnych cechach — najwyższe metryki, najlepiej radzi sobie z danymi.
6. **Charakter danych:**
   - Dane mają charakter nieliniowy, co potwierdza przewaga modeli drzewiastych nad liniowymi.

**Wnioski:**
- Modele drzewiaste (Random Forest) są rekomendowane do tego typu danych.
- Najważniejsze cechy to: saldo, wiek, status materialny, cechy kampanii i binarne wskaźniki finansowe.
- Transformacje należy akceptować tylko, jeśli realnie poprawiają metryki na walidacji.
- W tym przypadku oryginalne cechy okazały się lepsze niż rozbudowane feature engineering.

In [15]:
# Najważniejsze cechy dla Random Forest na oryginalnych cechach
feat_imp_rf_o = pd.Series(rf_o.feature_importances_, index=X_orig_enc.columns).sort_values(ascending=False)
print('Top 5 cech (Random Forest, oryginalne cechy):')
display(feat_imp_rf_o.head(5))

# Najważniejsze cechy dla Random Forest na nowych cechach
feat_imp_rf_n = pd.Series(rf_n.feature_importances_, index=X_new_enc.columns).sort_values(ascending=False)
print('Top 5 cech (Random Forest, nowe cechy):')
display(feat_imp_rf_n.head(5))

Top 5 cech (Random Forest, oryginalne cechy):


balance             0.199667
age                 0.171828
day                 0.153346
pdays               0.051996
poutcome_success    0.042386
dtype: float64

Top 5 cech (Random Forest, nowe cechy):


log_balance        0.221303
balance_shifted    0.220740
day                0.191274
age_squared        0.120664
age                0.119478
dtype: float64

### Wnioski

- W przypadku oryginalnych cech, model Random Forest wskazuje, że najważniejsze predyktory to zwykle saldo (balance), historia kontaktu (previous, pdays), status materialny oraz wybrane cechy demograficzne (np. job, age).
- W przypadku nowych cech (po transformacjach), ranking najważniejszych cech może się zmienić – często pojawiają się cechy pochodne (np. balance_bin, log_balance), ale ich wpływ na skuteczność modelu okazał się mniejszy niż oryginalnych zmiennych.
- Najwyższe metryki (AUC, F1) uzyskano dla modeli opartych na oryginalnych cechach, co potwierdza, że rozbudowane feature engineering nie zawsze poprawia jakość predykcji – czasem wręcz ją obniża.
- Analiza feature importance pozwala lepiej zrozumieć, które cechy są kluczowe dla decyzji modelu i powinna być zawsze wykonywana po treningu, by uzasadnić wybór zmiennych w końcowym modelu.
- Wnioski te są spójne z wcześniejszymi obserwacjami: prostszy, dobrze przygotowany zestaw cech często daje lepsze rezultaty niż nadmiernie rozbudowane transformacje.